In [1]:
# Part 5 - Spark MLlib demo (BFT65B)
!apt-get -qq update > /dev/null
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q install pyspark==3.5.3
import glob, os
cands = sorted(glob.glob("/usr/lib/jvm/java-17-openjdk*")) or sorted(glob.glob("/usr/lib/jvm/java-21-openjdk*"))
os.environ["JAVA_HOME"] = cands[0]
import pyspark
print("JAVA_HOME =", os.environ["JAVA_HOME"], "| PySpark", pyspark.__version__, "| CPU cores:", os.cpu_count())

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.
JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64 | PySpark 3.5.3 | CPU cores: 2


In [3]:
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Bigdata_final_project"
!cp "$BASE/mllib_input.parquet" /content/mllib_input.parquet
!ls -la /content/mllib_input.parquet

Mounted at /content/drive
-rw------- 1 root root 219137093 Nov 24 08:42 /content/mllib_input.parquet


In [4]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[*]").appName("bft65b-mllib")
         .config("spark.driver.memory", "8g").getOrCreate())
df = spark.read.parquet("/content/mllib_input.parquet")
FEATURES = [c for c in df.columns if c.startswith("F_")]
df = df.withColumn("label", F.col("Y").cast("double"))
for c in FEATURES:
    df = df.withColumn(c, F.col(c).cast("double"))
dev = df.filter(F.col("SPLIT") == "dev").cache()
test = df.filter(F.col("SPLIT") == "test").cache()
print(len(FEATURES), "features | dev", dev.count(), "rows,", int(dev.agg(F.sum("label")).first()[0]), "positive",
      "| test", test.count(), "rows,", int(test.agg(F.sum("label")).first()[0]), "positive")

20 features | dev 1195009 rows, 10261 positive | test 435446 rows, 2181 positive


In [5]:
import time
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import average_precision_score, roc_auc_score

imp_cols = [c + "_imp" for c in FEATURES]
pipe = Pipeline(stages=[
    Imputer(inputCols=FEATURES, outputCols=imp_cols, strategy="median"),          # medians learned on dev only
    VectorAssembler(inputCols=imp_cols, outputCol="raw"),
    StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True),  # mean/sd learned on dev only
    LogisticRegression(featuresCol="features", labelCol="label", maxIter=100),
])
t0 = time.perf_counter()
model = pipe.fit(dev)                                   # the whole pipeline is fitted on the development period
fit_s = time.perf_counter() - t0
pred = model.transform(test)                            # test is only transformed and scored, once

ev_pr = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")
ev_roc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
pdf = pred.select(vector_to_array("probability")[1].alias("p"), "label").toPandas()
res = pd.DataFrame([{
    "engine": "Spark MLlib logistic (local[*])", "fit_seconds_on_dev": round(fit_s, 1),
    "test_areaUnderPR_spark": ev_pr.evaluate(pred), "test_areaUnderROC_spark": ev_roc.evaluate(pred),
    "test_average_precision_sklearn": average_precision_score(pdf["label"], pdf["p"]),
    "test_roc_auc_sklearn": roc_auc_score(pdf["label"], pdf["p"]),
}])
res.to_csv("/content/part5_mllib_result.csv", index=False)
res.T

,0
engine,Spark MLlib logistic (local[*])
fit_seconds_on_dev,100.0
test_areaUnderPR_spark,0.369045
test_areaUnderROC_spark,0.953661
test_average_precision_sklearn,0.378853
test_roc_auc_sklearn,0.953672


In [6]:
!cp /content/part5_mllib_result.csv "$BASE/"
!ls -la "$BASE"
print("Saved. Now: File -> Save, then File -> Download -> Download .ipynb")

total 274403
drwx------  2 root root      4096 Nov 24 08:47 .
drwx------ 11 root root      4096 Nov 24 08:42 ..
-rw-------  1 root root 219137093 Nov 24 08:26 mllib_input.parquet
-rw-------  1 root root       521 Nov 23 17:55 part3_spark_check.csv
-rw-------  1 root root     40074 Nov 23 17:55 part3_spark_explain.txt
-rw-------  1 root root      2626 Nov 23 10:58 part3_spark_reference_by_year.csv
-rw-------  1 root root       227 Nov 23 17:55 part3_spark_timing.csv
-rw-------  1 root root       241 Nov 24 08:47 part5_mllib_result.csv
-rw-------  1 root root  61797076 Nov 23 10:58 spark_input.parquet
Saved. Now: File -> Save, then File -> Download -> Download .ipynb
